---
execute:
    echo: false
---

# Elite Dangerous Database Reader
> Read Elite Dangerous database files

In [ ]:
#| default_exp eddb.readers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import sys, time, logging, json, gzip, os, io
import pandas as pd
import edcompanion.core

from pathlib import Path
from edcompanion.core import configuration


In [ ]:
from confproxy.core import init_console_logging


In [ ]:
init_console_logging(__name__)

2026-02-11T02:41:32+0100 INFO	230616	__main__	core.py	init_console_logging	40	Installed <StreamHandler stderr (INFO)> for __main__


<Logger __main__ (INFO)>

In [ ]:
#| exporti
syslog = logging.getLogger(__name__)
syslog.info(f"Loading module {__name__}")

2026-02-11T02:41:33+0100 INFO	230616	__main__	2923348250.py	<module>	3	Loading module __main__


In [ ]:
eddb_conf = configuration['EDDB']

### File Reader as generator

In [ ]:
#| export
def dbfilereader(filename):
    """
        Opens 'filename' as generator for eddb style objects
    """

    chunksize = 64 * 1024 * 1024

    with gzip.open(filename, 'rt') as jsonfile:

        while True:
            chunk = jsonfile.readlines(chunksize)
            if chunk:
                for line in chunk:
                    if len(line) < 5:
                        continue

                    yield json.loads(line.rstrip(',\n\r '))

            else:
                break




In [ ]:
eddb_conf['systems_1day'] = "systems_1day.json.gz"

In [ ]:
systems1 = os.path.join(eddb_conf['local_dumps'], eddb_conf['galaxy_1day'])
print(systems1)

C:\Users\fenke\Saved Games\Data\galaxy_1day.json.gz


In [ ]:
os.path.exists(systems1)

True

In [ ]:
i = 0
for item in dbfilereader(systems1):
    if i == 4:
        print(json.dumps(item, indent=3))
        break
    i+=1


{
   "id64": 7996342,
   "name": "Footie AA-A g0",
   "coords": {
      "x": -30620.1875,
      "y": 171.40625,
      "z": 51702.0625
   },
   "allegiance": null,
   "government": "None",
   "primaryEconomy": "None",
   "secondaryEconomy": "None",
   "security": "Anarchy",
   "population": 0,
   "bodyCount": 9,
   "date": "2025-12-17 23:08:25+00",
   "bodies": [
      {
         "id64": 36028797026960310,
         "bodyId": 1,
         "name": "Footie AA-A g0 barycentre 1",
         "type": "Barycentre",
         "orbitalPeriod": 394580.630516564,
         "semiMajorAxis": 60.3509646372546,
         "orbitalEccentricity": 0.084358,
         "orbitalInclination": -40.982496,
         "argOfPeriapsis": 243.610332,
         "meanAnomaly": 345.943896,
         "ascendingNode": 117.477935,
         "timestamps": {
            "meanAnomaly": "2025-12-17T23:08:26Z"
         },
         "stations": [],
         "updateTime": "2025-12-17 23:08:26+00"
      },
      {
         "id64": 7205759404

### File Reader as chunked processor

In [ ]:
#| export

def dbfile_process(filename, process_chunk):
    """Opens file and calls process_chunk to process batches of items"""

    chunksize = 16 * 1024 * 1024

    with gzip.open(filename, 'rt') as jsonfile:

        while True:
            chunk = jsonfile.readlines(chunksize)
            if chunk:
                data = []
                for line in chunk:
                    if len(line) < 5:
                        continue

                    item = json.loads(line.rstrip(',\n\r '))
                    data.append(item)

                process_chunk(data)

            else:
                break



### File reader using pandas

In [ ]:
? gzip

Type:        module
String form: <module 'gzip' from '/usr/lib64/python3.11/gzip.py'>
File:        /usr/lib64/python3.11/gzip.py
Docstring:  
Functions that read and write gzipped files.

The user of the file doesn't have to worry about the compression,
but random access is not allowed.

In [ ]:
? gzip.open

Signature:
 gzip.open(
    filename,
    mode='rb',
    compresslevel=9,
    encoding=None,
    errors=None,
    newline=None,
)
Docstring:
Open a gzip-compressed file in binary or text mode.

The filename argument can be an actual filename (a str or bytes object), or
an existing file object to read from or write to.

The mode argument can be "r", "rb", "w", "wb", "x", "xb", "a" or "ab" for
binary mode, or "rt", "wt", "xt" or "at" for text mode. The default mode is
"rb", and the default compresslevel is 9.

For binary mode, this function is equivalent to the GzipFile constructor:
GzipFile(filename, mode, compresslevel). In this case, the encoding, errors
and newline arguments must not be provided.

For text mode, a GzipFile object is created, and wrapped in an
io.TextIOWrapper instance with the specified encoding, error handling
behavior, and line ending(s).
File:      /usr/lib64/python3.11/gzip.py
Type:      function

In [ ]:
? io.TextIOWrapper.readlines

Signature:  io.TextIOWrapper.readlines(self, hint=-1, /)
Docstring:
Return a list of lines from the stream.

hint can be specified to control the number of lines read: no more
lines will be read if the total size (in bytes/characters) of all
lines so far exceeds hint.
Type:      method_descriptor

In [ ]:
#| export

def dbfile_process_dataframes(filename, process_chunk):
    """Opens file and calls process_chunk to process batches of items"""

    chunksize = 128 * 1024

    with gzip.open(filename, 'rt') as jsonfile:

        while True:
            chunk = jsonfile.readlines(chunksize)
            if chunk:

                df = pd.read_json()

                process_chunk(data)

            else:
                break



### File Reader as async chunked processor

In [ ]:
#| export

async def dbfile_process_async(filename, process_chunk):
    """Opens file and calls process_chunk to process batches of items"""

    chunksize = 16 * 1024 * 1024

    with gzip.open(filename, 'rt') as jsonfile:

        while True:
            chunk = jsonfile.readlines(chunksize)
            if chunk:
                data = []
                for line in chunk:
                    if len(line) < 5:
                        continue

                    item = json.loads(line.rstrip(',\n\r '))
                    data.append(item)

                await process_chunk(data)

            else:
                break



### File Reader through pandas

In [ ]:
def 

In [ ]:
#| hidey
import nbdev; nbdev.nbdev_export()